# 04 · Baseline

How good is a simple guess, and how much do real models add? Four reference models on the six cities.
**None of them is a final model**: they set the bar that the tuned models, fit on the feature lists from
notebook 05, have to beat.

- Baseline 0: the city's median price, the same guess for every listing.
- Baseline 1: the median price of listings in the same neighbourhood and room type.
- Reference Elastic net: untuned, on all candidate features.
- Reference random forest: untuned, on the same features.

Data: `strfair.prepare()`, so the features include Kandice's 5 review sentiment columns
(`sentiment_review_count` is dropped, `no_sentiment` flags listings with fewer than 3 usable reviews). The
first run builds the sentiment scores and is slow: about 45 minutes for LA and 15 for Toronto.

Setup: 2026-09 snapshots, GroupKFold(5) by host (one host's listings stay in the same fold), MAE on log
price. Notebook 05 compares its feature lists against these numbers.

## 1. Setup

In [1]:
import sys
sys.path.insert(0, "..")   # ".." is the folder above notebooks/, i.e. the repo root, where strfair/ lives
import strfair             # the team's shared data pipeline

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold

# the September 2026 snapshot of each city (the newest one with prices)
SNAPSHOTS = {"new-york-city": "2026-09-14", 
                "los-angeles": "2026-09-14", 
                "toronto": "2026-09-16", 
                "san-diego": "2026-09-20",
                "clark-county-nv": "2026-09-20", 
                "jersey-city": "2026-09-21"}

SEED, N_SPLITS = 42, 5 # 

## 2. Baseline 0 and Baseline 1 (six cities)

**Baseline 0:** predict the median log price of the training listings for every listing. No features are
used, so any real model should beat this.

Why the median?

- If we predict one number for every listing, the median gives the lowest MAE.
- A few very expensive listings pull the mean up, but not the median.
- The median of log price is the log of the median price, so the guess is just the typical nightly price.

**Baseline 1:** for each listing, predict the median log price of the training listings that share its
neighbourhood and room type.

Why neighbourhood and room type?

- This is roughly how hosts price: look at similar places nearby.
- Room type and neighbourhood are two of the strongest single columns in `docs/features.md`.
- It uses only two columns, so the gap to the real models shows what the other features add.

Small groups: a group with fewer than 5 training listings is too noisy, so those listings get the
neighbourhood median, then the room type median, then the city median.

Reading the MAE: it is on log price, so an MAE of 0.68 means the guess is typically off by about 2x
(e^0.68 ≈ 2), and 0.47 means about 60%.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*not a team snapshot.*")   # we use 2026-09, not the pinned 2026-06

# columns that are never inputs: LEAKY and DEMAND_RESPONSE (strfair/features.py), the ids, the unpooled
# neighbourhood, and discount_amount, the quote's discount in dollars, which is also computed from the price
# (≈ price × quote_nights × discount_share / (1 − discount_shdouare)). sentiment_review_count is a review count
# under another name, so it goes too (same list as notebook 05)
drop = strfair.LEAKY + strfair.DEMAND_RESPONSE + strfair.ID_COLUMNS + ["neighbourhood_raw", "discount_amount",
                                                                         "sentiment_review_count"]

# 5 folds split by host: all listings of one host land in the same fold, so the model never sees a host's
# other (often near-identical) listings while it is tested on one of them. Same folds on every run.
gkf = GroupKFold(n_splits=5) 

# Baseline 0: one guess for everyone, the median log price of the training folds
def baseline0(X, y, groups):
    maes = []
    for train_idx, test_idx in gkf.split(X, y, groups):
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]
        pred = np.full(len(y_test), y_train.median())
        maes.append(mean_absolute_error(y_test, pred))
    return np.mean(maes)    # average MAE over the 5 folds


# Baseline 1: median log price of training listings with the same neighbourhood and room type
def baseline1(X, y, groups):
    maes = []
    for train_idx, test_idx in gkf.split(X, y, groups):
        X_train = X.iloc[train_idx] 
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # one label per listing, e.g. "Downtown | Entire home/apt"
        group_train = X_train["neighbourhood"] + " | " + X_train["room_type"]
        group_test = X_test["neighbourhood"] + " | " + X_test["room_type"]

        # median of each group with at least 5 listings
        counts = group_train.value_counts()
        big = group_train.isin(counts[counts >= 5].index)
        group_median = y_train[big].groupby(group_train[big]).median()

        # small or unseen groups: neighbourhood, then room type, then city median
        pred = group_test.map(group_median)
        pred = pred.fillna(X_test["neighbourhood"].map(y_train.groupby(X_train["neighbourhood"]).median()))
        pred = pred.fillna(X_test["room_type"].map(y_train.groupby(X_train["room_type"]).median()))
        pred = pred.fillna(y_train.median())

        maes.append(mean_absolute_error(y_test, pred))
    return np.mean(maes)


rows = []
for city, snapshot in SNAPSHOTS.items():
    data = strfair.prepare(city, snapshot)     # read from the cache after the first run
    F = data.features
    X = F[[c for c in F.columns if c not in drop]].copy()
    X["no_sentiment"] = X["avg_sentiment"].isna().astype(int)   # 1 = fewer than 3 usable reviews
    y = F["log_price"]                         # target: log of the nightly price
    groups = data.sample["host_id"].values     # used by GroupKFold to keep each host in one fold

    b0 = baseline0(X, y, groups)
    b1 = baseline1(X, y, groups)
    rows.append([city, len(y), X.shape[1], b0, b1])

results = pd.DataFrame(rows, columns=["city", "listings", "features", "baseline0_mae", "baseline1_mae"])
results["improvement"] = 1 - results["baseline1_mae"] / results["baseline0_mae"]   # share of Baseline 0's error removed
results.round(3)

new-york-city 2026-09-14 | listings 29,918 rows, 19,396 in sample | features 116 cols | code a07f95f6 | data 8d904721  (cached)


/Users/krystal/Desktop/GeorgiaTech/Courses/26Fall/CSE6242_DVA/Project/cse6242-airbnb/notebooks/../strfair/features.py:323: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["price_per_person"] = out["price"] / out["accommodates"].where(out["accommodates"] > 0)


los-angeles 2026-09-14 | listings 43,781 rows, 36,663 in sample | features 116 cols | code a07f95f6 | data c42b65f1


/Users/krystal/Desktop/GeorgiaTech/Courses/26Fall/CSE6242_DVA/Project/cse6242-airbnb/notebooks/../strfair/features.py:323: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["price_per_person"] = out["price"] / out["accommodates"].where(out["accommodates"] > 0)


toronto 2026-09-16 | listings 22,050 rows, 16,760 in sample | features 115 cols | code a07f95f6 | data 93c4b714
san-diego 2026-09-20 | listings 13,253 rows, 11,614 in sample | features 115 cols | code a07f95f6 | data 23c71b6d  (cached)
clark-county-nv 2026-09-20 | listings 20,651 rows, 17,440 in sample | features 115 cols | code a07f95f6 | data 2a715a57  (cached)
jersey-city 2026-09-21 | listings 2,330 rows, 2,093 in sample | features 115 cols | code a07f95f6 | data 460245b9  (cached)


,city,listings,features,baseline0_mae,baseline1_mae,improvement
0,new-york-city,19396,89,0.687,0.528,0.231
1,los-angeles,36663,89,0.710,0.524,0.262
2,toronto,16760,88,0.686,0.565,0.177
3,san-diego,11614,88,0.611,0.514,0.159
4,clark-county-nv,17440,88,0.641,0.592,0.077
5,jersey-city,2093,88,0.680,0.466,0.314


**Result** (2026-09 snapshots, GroupKFold by host)

- Baseline 1 beats Baseline 0 in every city.
- Clark County gains the least (8%): like Jersey City, its neighbourhoods are a few large areas with little
  price information. These two cities will need better location features.
- LA and NYC have many small neighbourhoods, so the neighbourhood + room type median helps more there.

## 3. Reference Elastic net

Linear model on all candidate features in `X` (the `features` column of the table in section 2), with the settings from
`notebooks/00`. No tuning yet: this is the reference that feature selection is compared against.

**Why Elastic net?**

- It is the interpretable model in `docs/models.md`: a coefficient is roughly a % change in price.
- The L1 + L2 penalty copes with many correlated features and can shrink weak ones to zero.

**Preprocessing** (refit on the training rows of every fold): numbers get the median for missing values
and are scaled; categories become one column per level, with levels under 20 listings grouped.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# a pipeline, so the median, scaling and one-hot levels are learned from the training folds only
def make_enet(X):
    cat = list(X.select_dtypes(exclude="number").columns)   # text columns, e.g. room_type, neighbourhood
    num = [c for c in X.columns if c not in cat]

    # numbers: fill missing values with the median, then scale
    # categories: one column per level, levels with < 20 listings are grouped together
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat),
    ])
    # alpha: overall penalty strength; l1_ratio=0.5: half L1 (can set weights to 0), half L2. Not tuned yet
    return make_pipeline(pre, ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000))

## 4. Reference random forest

Many decision trees on the same features, averaged. Also untuned: 300 trees, at least 5 listings per leaf.

**Why random forest?**

- It is the ensemble model in `docs/models.md`.
- It picks up non-linear effects and interactions (for example latitude with longitude) that a linear
  model misses, so the gap to Elastic net shows how much accuracy the interpretable model gives up.

**Preprocessing:** trees need no scaling or one-hot columns, so each category is just numbered.

In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder


def make_rf(X):
    cat = list(X.select_dtypes(exclude="number").columns)
    num = [c for c in X.columns if c not in cat]

    # trees don't need scaling or one-hot columns: each category just gets a number
    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)), cat),
    ])
    # 300 trees; a leaf needs at least 5 listings so a tree can't memorise single prices.
    # n_jobs=-1 uses every CPU core, random_state fixes the result
    return make_pipeline(pre, RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                    n_jobs=-1, random_state=42))

## 5. All four reference models (six cities)

In [5]:
# baseline0, baseline1, drop and gkf come from section 2; make_enet and make_rf from sections 3 and 4

# same 5 folds as the baselines, so all four models are scored on exactly the same listings
def run_model(make_model, X, y, groups):
    maes = []
    for train_idx, test_idx in gkf.split(X, y, groups):
        model = make_model(X)    # a new model each fold
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = model.predict(X.iloc[test_idx])
        maes.append(mean_absolute_error(y.iloc[test_idx], pred))
    return np.mean(maes)


rows = []
for city, snapshot in SNAPSHOTS.items():
    # same X, y and groups as section 2
    data = strfair.prepare(city, snapshot)
    F = data.features
    X = F[[c for c in F.columns if c not in drop]].copy()
    X["no_sentiment"] = X["avg_sentiment"].isna().astype(int)   # 1 = fewer than 3 usable reviews
    y = F["log_price"]
    groups = data.sample["host_id"].values

    b0 = baseline0(X, y, groups)
    b1 = baseline1(X, y, groups)
    m1 = run_model(make_enet, X, y, groups)
    m2 = run_model(make_rf, X, y, groups)     # the slowest step
    rows.append([city, len(y), b0, b1, m1, m2])

# MAE on log price: 0.28 means the prediction is typically off by about 32% (e^0.28 ≈ 1.32)
results = pd.DataFrame(rows, columns=["city", "listings", "baseline0_mae", "baseline1_mae",
                                      "elastic_net_mae", "random_forest_mae"])
results.round(3)

new-york-city 2026-09-14 | listings 29,918 rows, 19,396 in sample | features 116 cols | code a07f95f6 | data 8d904721  (cached)
los-angeles 2026-09-14 | listings 43,781 rows, 36,663 in sample | features 116 cols | code a07f95f6 | data c42b65f1  (cached)
toronto 2026-09-16 | listings 22,050 rows, 16,760 in sample | features 115 cols | code a07f95f6 | data 93c4b714  (cached)
san-diego 2026-09-20 | listings 13,253 rows, 11,614 in sample | features 115 cols | code a07f95f6 | data 23c71b6d  (cached)
clark-county-nv 2026-09-20 | listings 20,651 rows, 17,440 in sample | features 115 cols | code a07f95f6 | data 2a715a57  (cached)
jersey-city 2026-09-21 | listings 2,330 rows, 2,093 in sample | features 115 cols | code a07f95f6 | data 460245b9  (cached)


,city,listings,baseline0_mae,baseline1_mae,elastic_net_mae,random_forest_mae
0,new-york-city,19396,0.687,0.528,0.295,0.279
1,los-angeles,36663,0.710,0.524,0.283,0.269
2,toronto,16760,0.686,0.565,0.250,0.246
3,san-diego,11614,0.611,0.514,0.282,0.276
4,clark-county-nv,17440,0.641,0.592,0.344,0.308
5,jersey-city,2093,0.680,0.466,0.226,0.229


**Result** (2026-09 snapshots, GroupKFold by host, MAE on log price)

- Both models cut the error roughly in half compared with Baseline 1, in every city.
- Random forest is a little better than Elastic net in 5 of 6 cities. The biggest gap is Clark County
  (0.344 vs 0.308), where the forest can use latitude and longitude directly. Elsewhere the gap is at most
  0.016, so the interpretable model gives up little accuracy.
- The sentiment features change MAE by at most 0.003 in any city (compared with the run without them), so
  they add almost nothing to the price models.